In [1]:
#import required libraries.

import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import re
from datetime import datetime
import json

# 1. Configure scraping target and currency settings
base_url = "https://books.toscrape.com/"

supported_currencies = {
    "GBP": "British Pound",
    "KES": "Kenyan Shilling",
    "USD": "US Dollar",
    "EUR": "Euro",
}

print("Supported currencies:", list(supported_currencies.keys()))
default_currency = input("Enter source currency (default GBP): ").upper() or "GBP"
new_currency = input("Enter target currency (default KES): ").upper() or "KES"

if default_currency not in supported_currencies or new_currency not in supported_currencies:
    print("Invalid currency! Defaulting to GBP -> KES")
    default_currency = "GBP"
    new_currency = "KES"

print(f"\nConverting from {default_currency} ({supported_currencies[default_currency]}) to {new_currency} ({supported_currencies[new_currency]})")

# 2. Fetch web page with error handling
soup = None
try:
    response = requests.get(base_url, timeout=10)
    response.raise_for_status()
    print(f"\nSuccessfully fetched {base_url} (Status: {response.status_code})")
    soup = BeautifulSoup(response.content, "html.parser")
except requests.exceptions.ConnectionError:
    print("Connection Error: Could not connect to the website. Check your internet.")
except requests.exceptions.Timeout:
    print("Timeout Error: The server took too long to respond.")
except requests.exceptions.HTTPError as err:
    print(f"HTTP Error: {err}")

# 3. Parse and extract book titles and prices
products = []

if soup:
    books = soup.select(".product_pod")
    for book in books:
        try:
            title_tag = book.select_one("h3 a")
            title = title_tag["title"] if title_tag else "N/A"

            price_tag = book.select_one(".price_color")
            raw_price = price_tag.get_text(strip=True) if price_tag else "N/A"

            if title != "N/A" and raw_price != "N/A":
                products.append({"title": title, "raw_price": raw_price})
                if len(products) >= 10:
                    break
        except Exception as e:
            print(f"Skipped a book due to error: {e}")
            continue

    print(f"Extracted {len(products)} books from the page.")
else:
    print("No page content to parse.")

# 4. Clean and normalize price data
cleaned_products = []

for product in products:
    raw = product["raw_price"]
    price_match = re.search(r"[\d.]+", raw.replace(",", ""))
    if price_match:
        try:
            price_float = float(price_match.group())
            cleaned_products.append({"title": product["title"], "original_price": price_float})
        except ValueError:
            print(f"Could not convert price: {raw}")
    else:
        print(f"No price found in: {raw}")

print(f"\n{len(cleaned_products)} books with valid prices.")

# 5. Fetch live currency exchange rates
exchange_rate = None

try:
    api_url = f"https://api.frankfurter.app/latest?from={default_currency}&to={new_currency}"
    api_response = requests.get(api_url, timeout=10)
    api_response.raise_for_status()
    rate_data = api_response.json()
    exchange_rate = rate_data["rates"][new_currency]
    print(f"Exchange Rate: 1 {default_currency} = {exchange_rate} {new_currency}")
    print(f"Rate Date: {rate_data['date']}")
except requests.exceptions.ConnectionError:
    print("Could not connect to the exchange rate API.")
except requests.exceptions.Timeout:
    print("Exchange rate API timed out.")
except requests.exceptions.HTTPError as err:
    print(f"HTTP Error from API: {err}")
except KeyError:
    print(f"Could not find rate for {default_currency} -> {new_currency}")

# 6. Convert prices to selected currency
conversion_success = False
if exchange_rate:
    for product in cleaned_products:
        product["converted_price"] = round(product["original_price"] * exchange_rate, 2)
    conversion_success = True
    print(f"\nConverted {len(cleaned_products)} prices from {default_currency} to {new_currency}")
else:
    print("This currency is not supported or the same as the default currency.")

# 7. Add timestamps
conversion_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
for product in cleaned_products:
    product["conversion_timestamp"] = conversion_time
print(f"Timestamp: {conversion_time}")

# 8. Display results in a formatted table
df = pd.DataFrame(cleaned_products)
df = df.rename(columns={
    "title": "Product Name",
    "original_price": f"Price ({default_currency})",
    "converted_price": f"Price ({new_currency})",
    "conversion_timestamp": "Converted At"
})

print(f"\n{'='*80}")
print(f"BOOKS TO SCRAPE - Book Prices ({default_currency} to {new_currency})")
print(f"Exchange Rate: 1 {default_currency} = {exchange_rate} {new_currency}")
print(f"{'='*80}")
display(df)

# 9. Save to CSV and JSON
df.to_csv("books_converted.csv", index=False)
print("Saved to books_converted.csv")

df.to_json("books_converted.json", orient="records", indent=2)
print("Saved to books_converted.json")

# 10. Plot original vs converted prices bar chart
if conversion_success:
    plot_df = df.head(10)
    product_names = [name[:25] + "..." if len(name) > 25 else name for name in plot_df["Product Name"]]
    original_prices = plot_df[f"Price ({default_currency})"]
    converted_prices = plot_df[f"Price ({new_currency})"]

    x = np.arange(len(product_names))
    width = 0.35

    fig, ax1 = plt.subplots(figsize=(14, 6))

    bars1 = ax1.bar(x - width/2, original_prices, width, label=f"Original ({default_currency})", color="#2196F3")
    ax1.set_ylabel(f"Price ({default_currency})", color="#2196F3")
    ax1.tick_params(axis="y", labelcolor="#2196F3")

    ax2 = ax1.twinx()
    bars2 = ax2.bar(x + width/2, converted_prices, width, label=f"Converted ({new_currency})", color="#FF9800")
    ax2.set_ylabel(f"Price ({new_currency})", color="#FF9800")
    ax2.tick_params(axis="y", labelcolor="#FF9800")

    ax1.set_xlabel("Books")
    ax1.set_xticks(x)
    ax1.set_xticklabels(product_names, rotation=45, ha="right", fontsize=8)
    ax1.set_title(f"Books To Scrape: Original ({default_currency}) vs Converted ({new_currency}) Prices\nRate: 1 {default_currency} = {exchange_rate} {new_currency}")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

    plt.tight_layout()
    plt.show()
else:
    print("Skipping chart because no conversion was performed.")

Supported currencies: ['GBP', 'KES', 'USD', 'EUR']

Converting from GBP (British Pound) to KES (Kenyan Shilling)

Successfully fetched https://books.toscrape.com/ (Status: 200)
Extracted 10 books from the page.

10 books with valid prices.
HTTP Error from API: 404 Client Error: Not Found for url: https://api.frankfurter.dev/v1/latest?from=GBP&to=KES
This currency is not supported or the same as the default currency.
Timestamp: 2026-04-13 23:16:19

BOOKS TO SCRAPE - Book Prices (GBP to KES)
Exchange Rate: 1 GBP = None KES


,Product Name,Price (GBP),Converted At
0,A Light in the Attic,51.77,2026-04-13 23:16:19
1,Tipping the Velvet,53.74,2026-04-13 23:16:19
2,Soumission,50.10,2026-04-13 23:16:19
3,Sharp Objects,47.82,2026-04-13 23:16:19
4,Sapiens: A Brief History of Humankind,54.23,2026-04-13 23:16:19
5,The Requiem Red,22.65,2026-04-13 23:16:19
6,The Dirty Little Secrets of Getting Your Dream...,33.34,2026-04-13 23:16:19
7,The Coming Woman: A Novel Based on the Life of...,17.93,2026-04-13 23:16:19
8,The Boys in the Boat: Nine Americans and Their...,22.60,2026-04-13 23:16:19
9,The Black Maria,52.15,2026-04-13 23:16:19


Saved to books_converted.csv
Saved to books_converted.json
Skipping chart because no conversion was performed.
